# Day 92: The project's first chart

**Goal:** I have *counted* the GH-GhaSP problem. Today I make it *visible*.

**The picture:** a boxplot is a "height photo" of each group. The line in the box is the **median**, the box covers the **middle half** of the values, the whiskers reach the normal extremes, and the dots are unusual values.

> Restart the kernel, then run top to bottom.

## 0. Setup

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat, quality_flags, dataset_alarm, cleaning_report
from db_config import run_query
import matplotlib.pyplot as plt

## 1. Three small ideas
1. **pandas draws it in one line:** `df.boxplot(column=..., by=..., ax=ax)`. Add `plt.suptitle("")` to remove pandas' extra automatic title.
2. **Label everything.** Title (a finding), x-label, y-label with the unit.
3. **Save as a picture:** `fig.savefig("name.png", dpi=150, bbox_inches="tight")`.

## 2. Exercise on made-up data
**Prediction before drawing:** A and B near the bottom, C far above. With a **log scale** the A and B boxes spread out (each gridline is 10 times the one below).

Medians: A = 5, B = 6, C = 110, so C is about 18 to 22 times higher. "About 20 times as high" is more honest than "20 times higher".

In [ ]:
toy = pd.DataFrame({
    "group": ["A", "A", "A", "B", "B", "B", "C", "C", "C"],
    "value": [4, 5, 6, 5, 6, 7, 100, 110, 120]
})

fig, ax = plt.subplots()
toy.boxplot(column="value", by="group", ax=ax)
plt.suptitle("")
ax.set_xlabel("Group")
ax.set_ylabel("value (log scale)")
ax.set_title("Group C is about 20 times as high as A and B")
ax.set_yscale("log")
fig.savefig("toy_chart.png", dpi=150, bbox_inches="tight")

**Check question: why must a title state the finding?**
A topic title ("Values by group") makes the reader do the work. A finding title tells them what to learn before they look at the boxes.

## 3. The real chart

In [ ]:
flat = build_flat("orgc", ["ph", "clay", "sand"])
checked = quality_flags(flat)
meta = run_query("SELECT layer_id, dataset_id FROM wosis_orgc WHERE country_name = 'Ghana'")
traced = pd.merge(checked, meta, on="layer_id", how="left")
print(len(traced))        # must be 1756

**The order trap:** `boxplot(by=...)` draws the groups in **alphabetical order** (AF-AfSP, GH-GhaSP, WD-ISIS, WD-NWAFU-SCS, WD-WISE). The `n` labels must be in the same order, or every box wears the wrong number. `value_counts().sort_index()` gives the same order.

**A range must not rest on a group of one:** `WD-NWAFU-SCS` has a single layer (n = 1), so its median is just that layer. I compare GH-GhaSP only with datasets that have at least 10 layers. The numbers in the title are computed in code and never typed.

In [ ]:
counts = traced["dataset_id"].value_counts().sort_index()
labels = [f"{name}\n(n={count})" for name, count in counts.items()]

medians = traced.groupby("dataset_id")["orgc"].median()
big = counts[counts >= 10].index                         # datasets with 10+ layers
others = medians[medians.index.isin(big) & (medians.index != "GH-GhaSP")]
gh = medians["GH-GhaSP"]
times_low = gh / others.max()                            # about 25
times_high = gh / others.min()                           # about 43
print(f"GH-GhaSP median {gh:.1f}; others {others.min():.1f} to {others.max():.1f}; "
      f"ratio {times_low:.0f} to {times_high:.0f}")

**Prediction for the two panels:** on the plain axis the four other datasets are squashed flat. The log panel makes them readable and comparable. The plain panel shows how big the gap is. The log panel shows how the small datasets compare. For this story, the log panel is better, because it makes the four normal datasets visible.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax in (ax1, ax2):
    traced.boxplot(column="orgc", by="dataset_id", ax=ax)
    ax.set_xticklabels(labels, rotation=20)               # rotate so names do not collide
    ax.set_xlabel("Dataset (n = number of layers)")
    ax.axhline(200, linestyle="--", color="gray")         # the flag threshold
plt.suptitle("")

ax1.set_title("On a plain axis, GH-GhaSP squashes\nthe others flat", fontsize=10)
ax1.set_ylabel("Organic carbon (g/kg)")
ax1.text(2.6, 215, "flag threshold (200)", fontsize=8, color="gray")

ax2.set_yscale("log")
ax2.set_title(f"GH-GhaSP median ({gh:.1f} g/kg) is {times_low:.0f} to {times_high:.0f} times\n"
              f"the medians of datasets with 10+ layers ({others.min():.1f} to {others.max():.1f})",
              fontsize=10)
ax2.set_ylabel("Organic carbon (g/kg, log scale)")        # say it: a log axis can mislead
ax2.text(2.6, 250, "flag threshold (200)", fontsize=8, color="gray")

fig.savefig("../../Figures/day_92_orgc_by_dataset.png", dpi=150, bbox_inches="tight")

Then add this line to `csa_soil_project/README.md` (and to the case study):
`![Organic carbon by dataset](Figures/day_92_orgc_by_dataset.png)`

**Check question: why put `n = ...` under each box?**
Without it, `WD-NWAFU-SCS` would look like a flat line, and a reader would not know whether it stands for 1,000 samples or 1. With `n=1`, they know not to trust it.

## 4. What I keep from today
- How to read a boxplot (median line, middle-half box, whiskers, dots).
- Titles state the finding, with numbers computed from the data.
- The label order must match the box order (alphabetical).
- Always show `n`, and never base a range on a group of one.
- Rotate crowded labels; say "log scale" on a log axis.
- The dashed threshold line shows what the flag means: only GH-GhaSP crosses it.